# 06 — Evaluación, promoción y exportación

Evalúa el generador retenido, crea una ficha y un manifiesto y bloquea la promoción si no supera los umbrales. El repositorio conserva el manifiesto; el adaptador canónico se respalda en S3 o en otro registro versionado.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "aulaTEX":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if (ROOT / "aulaTEX/aulatex_training").is_dir():
    from aulaTEX.aulatex_training.motor_training import latex_generation_checks, sha256_file
else:
    from scripts.aulatex_training.motor_training import latex_generation_checks, sha256_file

adapter = ROOT / "aulaTEX/models/local/motor-inteligente"
metrics_path = adapter / "metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.exists() else {}
example = r"\section{Introducción}Contexto.\section{Análisis temático}Producto.\section{Conclusiones}Síntesis."
checks = latex_generation_checks(example)
manifest = {
    "name": "motor-inteligente",
    "base_model": "Qwen/Qwen2.5-3B-Instruct",
    "artifact_uri": "s3://CONFIGURE_BUCKET/motor-inteligente/CONFIGURE_VERSION/",
    "local_adapter_exists": adapter.is_dir(),
    "metrics": metrics,
    "checks_example": checks,
    "promoted": False,
}
out = ROOT / "aulaTEX/models/manifests/motor-inteligente-candidate.json"
out.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Configure S3 y complete evaluación humana antes de cambiar promoted a true.")